# PhoBERT — UIT-VSFC, same protocol as the local baseline

Notebook này chỉ gọi CLI trong code bundle; không lặp lại thuật toán training.

1. Colab: chọn Runtime → GPU; Kaggle: bật accelerator GPU và Internet.
2. Upload gói phobert-colab-20261107.zip. Dataset snapshot trong gói đã được kiểm định tại local.
3. Cài preset, kiểm tra CUDA, chọn nơi lưu run và train lần lượt hai task.
4. Tải hai result ZIP về local để so sánh. Chưa có kết quả training nào trong notebook chưa chạy này.

Không dùng test để điều chỉnh preset. Không chép checkpoint PhoBERT vào production.joblib.


In [ ]:
from pathlib import Path
import json, zipfile, hashlib, subprocess, sys, os
from datetime import datetime, timezone

BUNDLE_NAME = 'phobert-colab-20261107.zip'
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    uploaded = files.upload()
    if BUNDLE_NAME not in uploaded:
        raise ValueError('Upload exactly ' + BUNDLE_NAME)
    bundle_path = Path(BUNDLE_NAME).resolve()
    workspace = Path('/content/vsfc-workspace')
else:
    matches = list(Path('/kaggle/input').rglob(BUNDLE_NAME))
    if len(matches) != 1:
        raise ValueError('Add the code ZIP as a Kaggle input dataset, then rerun this cell.')
    bundle_path = matches[0]
    workspace = Path('/kaggle/working/vsfc-workspace')

if workspace.exists() and any(workspace.iterdir()):
    raise FileExistsError('Workspace already contains files. Use a fresh session or a new workspace path.')
workspace.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(bundle_path) as archive:
    for item in archive.infolist():
        target = (workspace / item.filename).resolve()
        if not target.is_relative_to(workspace.resolve()):
            raise ValueError('Unsafe archive member')
    archive.extractall(workspace)

handoff = json.loads((workspace / 'handoff.json').read_text())
for name, expected in handoff['files_sha256'].items():
    actual = hashlib.sha256((workspace / name).read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError('Bundle checksum mismatch: ' + name)
print('Bundle and dataset snapshot verified:', handoff['dataset_revision'])


## Install and runtime preflight

Cài vào GPU runtime, không cài đè môi trường baseline Windows. Training chạy bằng subprocess mới để tránh kernel giữ thư viện cũ sau pip. Nếu preflight lỗi, dừng và lưu thông báo; chưa train hay đổi hyperparameters để chữa lỗi môi trường.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                str(workspace / 'notebooks/requirements-colab.txt')], check=True)
subprocess.run([sys.executable, '-c',
    "import torch; from pyvi import ViTokenizer; "
    "assert torch.cuda.is_available(), 'GPU required: select a CUDA runtime'; "
    "print(torch.cuda.get_device_name(0)); "
    "print(ViTokenizer.tokenize('Giảng viên hướng dẫn nhiệt tình'))"], check=True)
subprocess.run([sys.executable, '-m', 'ml', '--help'], cwd=workspace, check=True)


## Persist results before training

Colab mặc định mount Google Drive để checkpoints không mất khi runtime ngắt. Việc mount được bạn cấp quyền trong notebook. Nếu tắt Drive, outputs chỉ nằm trong runtime và phải download trước khi kết thúc. Kaggle cần Save Version để giữ working outputs. Mỗi lần chạy dùng tên mới; không xóa hoặc ghi đè run cũ.


In [ ]:
USE_GOOGLE_DRIVE = True
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    runs_root = Path('/content/drive/MyDrive/vsfc-experiments')
else:
    runs_root = workspace / 'artifacts/runs'
runs_root.mkdir(parents=True, exist_ok=True)
run_tag = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
model_revision = handoff['model_revision']
completed_runs = {}


def train_task(task):
    attempt = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    run_dir = runs_root / ('phobert-' + task + '-' + attempt)
    command = [sys.executable, '-u', '-m', 'ml', 'train-phobert',
               '--data-dir', str(workspace / 'data/processed/uit-vsfc-v1'),
               '--run-dir', str(run_dir), '--task', task,
               '--model-revision', model_revision]
    log_path = runs_root / (run_dir.name + '.console.log')
    print('Run:', run_dir, flush=True)
    print('Console log:', log_path, flush=True)
    with log_path.open('x', encoding='utf-8') as log:
        with subprocess.Popen(command, cwd=workspace, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                              errors='replace', bufsize=1) as process:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end='', flush=True)
            return_code = process.wait()
    if return_code:
        raise RuntimeError(f'Training failed (exit {return_code}). Read {log_path}; export was not started.')
    metadata = json.loads((run_dir / 'run.json').read_text())
    if metadata['status'] != 'complete':
        raise RuntimeError('Training did not complete')
    completed_runs[task] = run_dir
    print('Completed', task, run_dir)


def export_task(task):
    run_dir = completed_runs[task]
    output = workspace / (run_dir.name + '-results.zip')
    if output.exists():
        raise FileExistsError(output)
    with zipfile.ZipFile(output, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in run_dir.rglob('*'):
            if not path.is_file():
                continue
            rel = path.relative_to(run_dir)
            if rel.parts[0] == 'best_model' or path.name in {
                    'run.json', 'metrics.json', 'predictions.json', 'trainer_state.json'}:
                archive.write(path, (Path(run_dir.name) / rel).as_posix())
    print('Result ZIP:', output)
    if IN_COLAB:
        files.download(str(output))
    return output


## Task 1 — sentiment

Giữ nguyên preset từ protocol; checkpoint tốt nhất được chọn bằng validation macro F1. Tải kết quả task này ngay khi xong.


In [ ]:
train_task('sentiment')
export_task('sentiment')


## Task 2 — topic

Dùng cùng dataset manifest và model revision. Tải kết quả sau khi hoàn thành, rồi đưa cả hai result ZIP về local để thực hiện model comparison.


In [ ]:
train_task('topic')
export_task('topic')


## Completion gate

Cần có hai completed run với run.json, metrics.json, predictions.json, best_model/tokenizer và Trainer state. Chưa chọn model thắng hoặc promote ở đây. Không phát hành notebook sau khi chạy nếu output hiển thị token, private paths hoặc dữ liệu nội bộ. Dataset trong bundle là UIT-VSFC công khai cho nghiên cứu; điều kiện phân phối ngoài đề tài vẫn cần được xác minh.
